# 06 — Real-Time Agent Orchestrator

This notebook turns the previously built Smart Twin + Maintenance Agent + Alert Agent into an **event-driven runtime simulation**.

The architecture is:

```text
IoT reading
     ↓
Smart Twin result
     ↓
Agent Orchestrator
     ↓
Is action required?
   ↙             ↘
 NO              YES
 ↓                ↓
continue      Maintenance Agent
monitoring         ↓
              Alert Agent
                   ↓
             operator alert
```

The IoT simulation already generates time-ordered readings for 7 machines. This notebook **replays those readings as if they were arriving in real time**.

Important:

- The ML models are not retrained.
- The Maintenance Agent and Alert Agent remain specialized agents.
- The Orchestrator decides **when those agents need to be invoked**.
- The loop is intentionally bounded so the notebook does not run forever.


In [1]:
# ============================================
# STEP 1: IMPORTS
# ============================================

import time
import pandas as pd
import numpy as np

from IPython.display import clear_output, display

print("Real-time Agent Orchestrator ready.")


Real-time Agent Orchestrator ready.


In [2]:
# ============================================
# STEP 2: LOAD SMART TWIN STREAM
# ============================================

input_path = "../data/processed/iot_smart_twin_results.csv"

stream_df = pd.read_csv(input_path)

stream_df = stream_df.sort_values(
    by=["Cycle", "Machine_ID"]
).reset_index(drop=True)

print("Smart Twin stream loaded.")
print("Rows:", len(stream_df))
print("Machines:", stream_df["Machine_ID"].nunique())

display(
    stream_df[
        [
            "Machine_ID",
            "Machine_Type",
            "Cycle",
            "failure_probability",
            "failure_risk",
            "predicted_rul_cycles",
            "overall_status"
        ]
    ].head(10)
)


Smart Twin stream loaded.
Rows: 1930
Machines: 7


,Machine_ID,Machine_Type,Cycle,failure_probability,failure_risk,predicted_rul_cycles,overall_status
0,CNC_001,CNC_Mill,1,0.0001,LOW,256.45,HEALTHY
1,COMP_003,Compressor,1,0.0001,LOW,284.57,HEALTHY
2,MOTOR_004,Electric_Motor,1,0.0001,LOW,238.10,HEALTHY
3,PRESS_006,Hydraulic_Press,1,0.0001,LOW,273.77,HEALTHY
4,PUMP_002,Hydraulic_Pump,1,0.0001,LOW,306.94,HEALTHY
5,ROBOT_007,Industrial_Robot,1,0.0001,LOW,224.29,HEALTHY
6,TURB_005,Turbine,1,0.0001,LOW,327.21,HEALTHY
7,CNC_001,CNC_Mill,2,0.0001,LOW,262.11,HEALTHY
8,COMP_003,Compressor,2,0.0001,LOW,285.62,HEALTHY
9,MOTOR_004,Electric_Motor,2,0.0001,LOW,238.74,HEALTHY


## Why we need an Orchestrator

The agents should not be called for every normal reading.

For example:

```text
Healthy reading
     ↓
Orchestrator
     ↓
No action
     ↓
continue
```

But when the Smart Twin produces a warning/critical state:

```text
Warning/Critical reading
     ↓
Orchestrator
     ↓
Maintenance Agent
     ↓
Alert Agent
```

This is the **orchestration layer**.


In [3]:
# ============================================
# STEP 3: MAINTENANCE AGENT
# ============================================

def maintenance_agent(row):

    failure_risk = row["failure_risk"]
    rul = float(row["predicted_rul_cycles"])
    status = row["overall_status"]
    health = float(row["health_score"])

    if (
        failure_risk == "HIGH"
        or rul <= 20
        or status == "CRITICAL"
        or health < 30
    ):
        return {
            "maintenance_priority": "URGENT",
            "maintenance_decision": "IMMEDIATE MAINTENANCE",
            "maintenance_reason": (
                "Critical machine condition detected."
            )
        }

    elif (
        failure_risk == "MEDIUM"
        or rul <= 50
        or status == "WARNING"
    ):
        return {
            "maintenance_priority": "HIGH",
            "maintenance_decision": "SCHEDULE INSPECTION",
            "maintenance_reason": (
                "Machine degradation requires inspection."
            )
        }

    else:
        return {
            "maintenance_priority": "ROUTINE",
            "maintenance_decision": "NORMAL OPERATION",
            "maintenance_reason": (
                "Machine is operating within the healthy range."
            )
        }

print("Maintenance Agent loaded.")


Maintenance Agent loaded.


In [4]:
# ============================================
# STEP 4: ALERT AGENT
# ============================================

def alert_agent(row, maintenance_result):

    probability = float(
        row["failure_probability"]
    )

    rul = float(
        row["predicted_rul_cycles"]
    )

    risk = row["failure_risk"]
    status = row["overall_status"]

    machine_id = row["Machine_ID"]
    machine_type = row["Machine_Type"]
    cycle = row["Cycle"]

    action = maintenance_result[
        "maintenance_decision"
    ]

    if (
        risk == "HIGH"
        or rul <= 20
        or status == "CRITICAL"
    ):
        level = "CRITICAL"
        required = True

        message = (
            f"CRITICAL ALERT | {machine_id} "
            f"({machine_type}) | Cycle {cycle} | "
            f"Failure probability={probability:.2%} | "
            f"RUL={rul:.2f} cycles | "
            f"Action={action}"
        )

    elif (
        risk == "MEDIUM"
        or rul <= 50
        or status == "WARNING"
    ):
        level = "WARNING"
        required = True

        message = (
            f"WARNING | {machine_id} "
            f"({machine_type}) | Cycle {cycle} | "
            f"Failure probability={probability:.2%} | "
            f"RUL={rul:.2f} cycles | "
            f"Action={action}"
        )

    else:
        level = "INFO"
        required = False

        message = (
            f"INFO | {machine_id} | "
            f"Operating normally."
        )

    return {
        "alert_required": required,
        "alert_level": level,
        "alert_message": message
    }

print("Alert Agent loaded.")


Alert Agent loaded.


## Orchestrator policy

The orchestrator watches the Smart Twin state.

It invokes the agents when:

1. A machine becomes `WARNING` or `CRITICAL`.
2. Failure risk becomes `MEDIUM` or `HIGH`.
3. Predicted RUL reaches 50 cycles or below.
4. A machine transitions from a healthy state into an actionable state.

Healthy readings are simply monitored.


In [5]:
# ============================================
# STEP 5: ORCHESTRATOR
# ============================================

def should_trigger_agents(row, previous_state):

    risk = row["failure_risk"]
    status = row["overall_status"]
    rul = float(row["predicted_rul_cycles"])

    actionable_now = (
        risk in ["MEDIUM", "HIGH"]
        or status in ["WARNING", "CRITICAL"]
        or rul <= 50
    )

    if not actionable_now:
        return False, "NORMAL"

    if previous_state is None:
        return True, "NEW_ACTIONABLE_STATE"

    previous_risk = previous_state["failure_risk"]
    previous_status = previous_state["overall_status"]
    previous_rul = float(
        previous_state["predicted_rul_cycles"]
    )

    risk_changed = (
        previous_risk != risk
    )

    status_changed = (
        previous_status != status
    )

    rul_crossed = (
        previous_rul > 50
        and rul <= 50
    )

    critical_crossed = (
        previous_rul > 20
        and rul <= 20
    )

    if (
        risk_changed
        or status_changed
        or rul_crossed
        or critical_crossed
    ):
        return True, "STATE_CHANGE"

    return False, "ALREADY_ACTIONABLE"


def run_orchestrator(row, previous_state):

    trigger, trigger_reason = (
        should_trigger_agents(
            row,
            previous_state
        )
    )

    if not trigger:

        return {
            "agents_invoked": False,
            "trigger_reason": trigger_reason,
            "maintenance_priority": "NONE",
            "maintenance_decision": "NO ACTION",
            "alert_level": "NONE",
            "alert_required": False,
            "alert_message": ""
        }

    maintenance_result = (
        maintenance_agent(row)
    )

    alert_result = alert_agent(
        row,
        maintenance_result
    )

    return {
        "agents_invoked": True,
        "trigger_reason": trigger_reason,
        **maintenance_result,
        **alert_result
    }

print("Orchestrator ready.")


Orchestrator ready.


In [6]:
# ============================================
# STEP 6: TEST ORCHESTRATOR ON A NORMAL EVENT
# ============================================

normal_row = stream_df[
    stream_df["overall_status"] == "HEALTHY"
].iloc[0]

normal_result = run_orchestrator(
    normal_row,
    None
)

print("NORMAL EVENT")
print("============")

for key, value in normal_result.items():
    print(f"{key}: {value}")


NORMAL EVENT
agents_invoked: False
trigger_reason: NORMAL
maintenance_priority: NONE
maintenance_decision: NO ACTION
alert_level: NONE
alert_required: False
alert_message: 


In [7]:
# ============================================
# STEP 7: TEST ORCHESTRATOR ON A CRITICAL EVENT
# ============================================

critical_row = stream_df[
    stream_df["overall_status"] == "CRITICAL"
].iloc[0]

critical_result = run_orchestrator(
    critical_row,
    None
)

print("CRITICAL EVENT")
print("==============")

for key, value in critical_result.items():
    print(f"{key}: {value}")


CRITICAL EVENT
agents_invoked: True
trigger_reason: NEW_ACTIONABLE_STATE
maintenance_priority: URGENT
maintenance_decision: IMMEDIATE MAINTENANCE
maintenance_reason: Critical machine condition detected.
alert_required: True
alert_level: CRITICAL
alert_message: CRITICAL ALERT | ROBOT_007 (Industrial_Robot) | Cycle 189 | Failure probability=0.10% | RUL=16.16 cycles | Action=IMMEDIATE MAINTENANCE


# Real-Time Simulation

The following loop replays the Smart Twin stream.

Each iteration represents a new incoming IoT time step.

For the demo we use a small delay such as **2 seconds**.

You can increase/decrease it with `SIMULATION_DELAY_SECONDS`.


In [8]:
# ============================================
# STEP 8: CONFIGURE REAL-TIME REPLAY
# ============================================

SIMULATION_DELAY_SECONDS = 0.1

# Number of simulation cycles to replay.
# None = replay the entire dataset.
MAX_CYCLES = None

print(
    "Simulation delay:",
    SIMULATION_DELAY_SECONDS,
    "seconds"
)

print(
    "Maximum cycles:",
    MAX_CYCLES
)


Simulation delay: 0.1 seconds
Maximum cycles: None


In [9]:
# ============================================
# STEP 9: REAL-TIME ORCHESTRATED LOOP
# ============================================

machine_state = {}

runtime_events = []

available_cycles = (
    sorted(
        stream_df["Cycle"]
        .unique()
    )
)

if MAX_CYCLES is not None:
    available_cycles = available_cycles[
        :MAX_CYCLES
    ]

print("Starting real-time agent simulation...")
print()

for cycle in available_cycles:

    cycle_rows = stream_df[
        stream_df["Cycle"] == cycle
    ]

    cycle_events = []

    for _, row in cycle_rows.iterrows():

        machine_id = row["Machine_ID"]

        previous_state = machine_state.get(
            machine_id
        )

        result = run_orchestrator(
            row,
            previous_state
        )

        # Save current state for future
        # transition detection.
        machine_state[machine_id] = {
            "failure_risk": row["failure_risk"],
            "overall_status": row["overall_status"],
            "predicted_rul_cycles": row[
                "predicted_rul_cycles"
            ]
        }

        event = {
            "Cycle": cycle,
            "Machine_ID": machine_id,
            "Machine_Type": row[
                "Machine_Type"
            ],
            "Simulation_Fault": row[
                "Simulation_Fault"
            ],
            "failure_probability": row[
                "failure_probability"
            ],
            "predicted_rul_cycles": row[
                "predicted_rul_cycles"
            ],
            "overall_status": row[
                "overall_status"
            ],
            **result
        }

        runtime_events.append(event)
        cycle_events.append(event)

    clear_output(wait=True)

    display_df = pd.DataFrame(
        cycle_events
    )

    print(
        f"REAL-TIME SIMULATION | Cycle {cycle}"
    )
    print(
        f"Machines processed: "
        f"{len(cycle_events)}"
    )

    display(
        display_df[
            [
                "Machine_ID",
                "Machine_Type",
                "failure_probability",
                "predicted_rul_cycles",
                "overall_status",
                "agents_invoked",
                "maintenance_decision",
                "alert_level"
            ]
        ]
    )

    actionable = display_df[
        display_df["agents_invoked"] == True
    ]

    if len(actionable) > 0:

        print("\nAGENT EVENTS")

        for _, event in actionable.iterrows():

            print(
                f"[{event['alert_level']}] "
                f"{event['Machine_ID']} → "
                f"{event['maintenance_decision']}"
            )

            if event["alert_message"]:
                print(
                    f"  {event['alert_message']}"
                )

    time.sleep(
        SIMULATION_DELAY_SECONDS
    )

print("\nReal-time simulation completed.")


REAL-TIME SIMULATION | Cycle 330
Machines processed: 1


,Machine_ID,Machine_Type,failure_probability,predicted_rul_cycles,overall_status,agents_invoked,maintenance_decision,alert_level
0,TURB_005,Turbine,0.9967,1.64,CRITICAL,False,NO ACTION,NONE



Real-time simulation completed.


In [10]:
# ============================================
# STEP 10: RUNTIME SUMMARY
# ============================================

runtime_df = pd.DataFrame(
    runtime_events
)

print("RUNTIME SUMMARY")
print("===============")

print(
    "Events processed:",
    len(runtime_df)
)

print(
    "Agent invocations:",
    int(
        runtime_df[
            "agents_invoked"
        ].sum()
    )
)

print(
    "Critical alerts:",
    int(
        (
            runtime_df["alert_level"]
            == "CRITICAL"
        ).sum()
    )
)

print(
    "Warnings:",
    int(
        (
            runtime_df["alert_level"]
            == "WARNING"
        ).sum()
    )
)

display(
    runtime_df[
        runtime_df["agents_invoked"] == True
    ].head(20)
)


RUNTIME SUMMARY
Events processed: 1930
Agent invocations: 55
Critical alerts: 35
Warnings: 20


,Cycle,Machine_ID,Machine_Type,Simulation_Fault,failure_probability,predicted_rul_cycles,overall_status,agents_invoked,trigger_reason,maintenance_priority,maintenance_decision,alert_level,alert_required,alert_message,maintenance_reason
1097,157,ROBOT_007,Industrial_Robot,ACTUATOR_DEGRADATION,0.0002,49.12,WARNING,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | ROBOT_007 (Industrial_Robot) | Cycle...,Machine degradation requires inspection.
1125,161,ROBOT_007,Industrial_Robot,ACTUATOR_DEGRADATION,0.0003,48.37,WARNING,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | ROBOT_007 (Industrial_Robot) | Cycle...,Machine degradation requires inspection.
1139,163,ROBOT_007,Industrial_Robot,ACTUATOR_DEGRADATION,0.0002,48.41,WARNING,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | ROBOT_007 (Industrial_Robot) | Cycle...,Machine degradation requires inspection.
1206,173,MOTOR_004,Electric_Motor,ELECTRICAL,0.0006,50.00,HEALTHY,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | MOTOR_004 (Electric_Motor) | Cycle 1...,Machine degradation requires inspection.
1241,178,MOTOR_004,Electric_Motor,ELECTRICAL,0.0006,48.63,WARNING,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | MOTOR_004 (Electric_Motor) | Cycle 1...,Machine degradation requires inspection.
1255,180,MOTOR_004,Electric_Motor,ELECTRICAL,0.0003,47.24,WARNING,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | MOTOR_004 (Electric_Motor) | Cycle 1...,Machine degradation requires inspection.
1321,189,ROBOT_007,Industrial_Robot,ACTUATOR_DEGRADATION,0.0010,16.16,CRITICAL,True,STATE_CHANGE,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,True,CRITICAL ALERT | ROBOT_007 (Industrial_Robot) ...,Critical machine condition detected.
1328,190,ROBOT_007,Industrial_Robot,ACTUATOR_DEGRADATION,0.7479,15.49,CRITICAL,True,STATE_CHANGE,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,True,CRITICAL ALERT | ROBOT_007 (Industrial_Robot) ...,Critical machine condition detected.
1335,191,ROBOT_007,Industrial_Robot,ACTUATOR_DEGRADATION,0.2371,17.26,CRITICAL,True,STATE_CHANGE,URGENT,IMMEDIATE MAINTENANCE,CRITICAL,True,CRITICAL ALERT | ROBOT_007 (Industrial_Robot) ...,Critical machine condition detected.
1344,193,CNC_001,CNC_Mill,BEARING_WEAR,0.0002,47.87,WARNING,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | CNC_001 (CNC_Mill) | Cycle 193 | Fai...,Machine degradation requires inspection.


In [11]:
# ============================================
# STEP 11: SAVE RUNTIME EVENTS
# ============================================

runtime_output_path = (
    "../data/processed/"
    "orchestrated_runtime_events.csv"
)

runtime_df.to_csv(
    runtime_output_path,
    index=False
)

print(
    "Runtime events saved to:"
)

print(
    runtime_output_path
)


Runtime events saved to:
../data/processed/orchestrated_runtime_events.csv


# What this notebook now represents

The runtime architecture is:

```text
                    IoT Simulation
                          ↓
                    New reading
                          ↓
                     Smart Twin
                          ↓
                    Orchestrator
                          ↓
                  ┌───────┴───────┐
                  │               │
              Normal          Abnormal
                  │               │
             continue       Maintenance
             monitoring         Agent
                                  ↓
                              Alert Agent
                                  ↓
                           Operator alert
```

The agents are therefore **event-driven**, not constantly reasoning.

The orchestrator remains active and watches the incoming machine states.

This is the bridge between our batch notebook architecture and a future real streaming system such as MQTT/Kafka.

## Next phase

Once this runtime loop works, we add the **LLM layer**:

```text
Orchestrator
     ↓
Maintenance / Alert Agents
     ↓
LLM
     ↓
RAG maintenance knowledge
     ↓
Human-readable explanation
```

The LLM will not replace the predictive ML models or the safety-oriented decision thresholds.
